In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    base_url="https://api.openai.com/v1",
    api_key="api"
)

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from pydantic import BaseModel
import json

class Questions(BaseModel):
    questions: list[str]

In [5]:
target_files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]

In [6]:
filtered_docs = [doc for doc in documents if doc.get("filename") in target_files]

ground_truth_records = []
tokens_list = []
usage_list = []

In [11]:
from evaluation_utils import llm_structured, calc_price

for doc in filtered_docs:
    filename = doc.get("filename")
    content = doc.get("content")
    
    user_prompt_data = {
        "filename": filename,
        "content": content
    }
    user_prompt_json = json.dumps(user_prompt_data)
    
    
    try:
        result, usage = llm_structured(
            openai_client,
            data_gen_instructions,
            user_prompt_json,
            Questions,
        )
        for q in result.questions:
            ground_truth_records.append({
                "filename": filename,
                "question": q
            })
            
        tokens_list.append(calc_price(usage))
        usage_list.append([usage.input_tokens, usage.output_tokens])
            
    except Exception as e:
        print(f"Error processing {filename}: {e}")

In [28]:
usage_list

[[1021, 131], [1287, 88], [1754, 98]]

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [9]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [39]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")


In [40]:
q = ground_truth[0]["question"]

In [68]:
from minsearch import VectorSearch, Index
from embedder import Embedder

embedder = Embedder()
chunk_texts = [chunk['content'] for chunk in chunks]
X = embedder.encode_batch(chunk_texts)

def text_search(query, num_results=10):
    tindex = Index(text_fields=["content"], keyword_fields=["filename"])
    tindex.fit(chunks)
    return tindex.search(query, num_results=num_results)

def vector_search(query, num_results=10):
    vindex = VectorSearch(keyword_fields=["filename"])
    vindex.fit(X, chunks)
    q_encode = embedder.encode(query)
    return vindex.search(q_encode, num_results=num_results)

In [54]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [16]:
result = text_search(q, k=10)
first_result_filename = result[0]["filename"]
first_result_filename

'01-agentic-rag/lessons/03-rag.md'

In [27]:
vector_results = vector_search(q, k=10)
vector_results = vector_results[0]["filename"]
vector_results

'01-agentic-rag/lessons/01-intro.md'

In [44]:
def compute_relevance(q, search_function):
    expected_filename = q["filename"]
    results = search_function(query=q["question"], k=10)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == expected_filename))

    return relevance

In [34]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [35]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)



In [45]:
relevance_total = compute_relevance_total(ground_truth, text_search)

100%|██████████| 360/360 [00:11<00:00, 32.67it/s]


In [46]:
hit_rate(relevance_total)

0.8416666666666667

In [47]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)


relevance1 = compute_relevance_total(ground_truth, vector_search)
mrr(relevance1)

100%|██████████| 360/360 [00:01<00:00, 253.25it/s]


0.5646472663139328

In [69]:
k_values = [1, 50, 100, 200]

for k_value in k_values:
    def current_hybrid_search(query, k=10):
        return hybrid_search(query, k=k_value)
    curr_relevance = compute_relevance_total(ground_truth, current_hybrid_search)
    current_mrr = mrr(curr_relevance)
    print(f"При k={k_value} | MRR: {current_mrr:.4f}")

100%|██████████| 360/360 [00:13<00:00, 26.16it/s]


При k=1 | MRR: 0.6482


100%|██████████| 360/360 [00:13<00:00, 26.24it/s]


При k=50 | MRR: 0.6379


100%|██████████| 360/360 [00:13<00:00, 26.38it/s]


При k=100 | MRR: 0.6379


100%|██████████| 360/360 [00:13<00:00, 26.47it/s]

При k=200 | MRR: 0.6379
